In [18]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report
from sklearn.metrics import precision_recall_fscore_support

# Load Dataset
data = pd.read_csv(
    r"C:\Users\swathi\OneDrive\Desktop\nlp skill\class task\pos_tags_large.csv"
)

print(data.head())
print(data.shape)

   sentence_id    word  tag
0            1     The   DT
1            1     big   JJ
2            1   child   NN
3            1    eats  VBZ
4            1  slowly   RB
(5000, 3)


In [19]:
# Create sentences
sentences = []

for sid, group in data.groupby("sentence_id"):

    sentence = []

    for _, row in group.iterrows():

        word = str(row["word"])
        tag = str(row["tag"])

        sentence.append((word, tag))

    sentences.append(sentence)

print("Total Sentences :", len(sentences))
print("\nFirst Sentence:")
print(sentences[0])

Total Sentences : 1000

First Sentence:
[('The', 'DT'), ('big', 'JJ'), ('child', 'NN'), ('eats', 'VBZ'), ('slowly', 'RB')]


In [20]:
train_data, test_data = train_test_split(
    sentences,
    test_size=0.2,
    random_state=42
)

print("Training Sentences :", len(train_data))
print("Testing Sentences :", len(test_data))

Training Sentences : 800
Testing Sentences : 200


In [21]:
vocabulary = set()
tags = set()

for sentence in train_data:

    for word, tag in sentence:

        vocabulary.add(word.lower())
        tags.add(tag)

vocabulary = list(vocabulary)
tags = list(tags)

word_to_index = {
    word: i
    for i, word in enumerate(vocabulary)
}

tag_to_index = {
    tag: i
    for i, tag in enumerate(tags)
}

index_to_tag = {
    i: tag
    for tag, i in tag_to_index.items()
}

V = len(vocabulary)
T = len(tags)

print("Vocabulary Size :", V)
print("Number of POS Tags :", T)

Vocabulary Size : 37
Number of POS Tags : 5


In [22]:
# Initial, Transition, Emission Matrices

initial = np.ones(T)

transition = np.ones((T, T))

emission = np.ones((T, V))

print("Initial Matrix Shape :", initial.shape)
print("Transition Matrix Shape :", transition.shape)
print("Emission Matrix Shape :", emission.shape)

Initial Matrix Shape : (5,)
Transition Matrix Shape : (5, 5)
Emission Matrix Shape : (5, 37)


In [23]:
# Count Initial, Transition and Emission Probabilities

for sentence in train_data:

    # Initial Probability
    first_tag = sentence[0][1]
    initial[tag_to_index[first_tag]] += 1

    # Transition & Emission
    for i, (word, tag) in enumerate(sentence):

        word = word.lower()
        tag_index = tag_to_index[tag]

        # Emission Count
        if word in word_to_index:
            emission[tag_index, word_to_index[word]] += 1

        # Transition Count
        if i > 0:
            previous_tag = sentence[i - 1][1]

            transition[
                tag_to_index[previous_tag],
                tag_index
            ] += 1

print("Counting Completed Successfully!")

Counting Completed Successfully!


In [24]:
# Convert Counts into Probabilities

initial = initial / initial.sum()

transition = transition / transition.sum(
    axis=1,
    keepdims=True
)

emission = emission / emission.sum(
    axis=1,
    keepdims=True
)

print("Probability Conversion Completed!")

Probability Conversion Completed!


In [25]:
# Convert to Log Space

log_initial = np.log(initial)

log_transition = np.log(transition)

log_emission = np.log(emission)

print("Log Space Conversion Completed!")

Log Space Conversion Completed!


In [26]:
print("Initial Matrix")
print(initial)

print("\nTransition Matrix Shape:", transition.shape)

print("\nEmission Matrix Shape:", emission.shape)

Initial Matrix
[0.99503106 0.00124224 0.00124224 0.00124224 0.00124224]

Transition Matrix Shape: (5, 5)

Emission Matrix Shape: (5, 37)


In [27]:
# -------------------------------
# Vectorized Viterbi Algorithm
# -------------------------------

def viterbi(sentence):

    n = len(sentence)

    dp = np.full((T, n), -np.inf)

    backpointer = np.zeros((T, n), dtype=int)

    # First Word
    word = sentence[0].lower()

    if word in word_to_index:
        emission_prob = log_emission[:, word_to_index[word]]
    else:
        emission_prob = np.log(np.ones(T) * 1e-10)

    dp[:, 0] = log_initial + emission_prob

    # Remaining Words
    for i in range(1, n):

        word = sentence[i].lower()

        if word in word_to_index:
            emission_prob = log_emission[:, word_to_index[word]]
        else:
            emission_prob = np.log(np.ones(T) * 1e-10)

        scores = dp[:, i-1].reshape(-1,1) + log_transition

        backpointer[:, i] = np.argmax(scores, axis=0)

        dp[:, i] = np.max(scores, axis=0) + emission_prob

    # Backtracking
    best_last = np.argmax(dp[:, -1])

    best_path = [best_last]

    for i in range(n-1,0,-1):
        best_last = backpointer[best_last, i]
        best_path.append(best_last)

    best_path.reverse()

    predicted_tags = [index_to_tag[i] for i in best_path]

    return predicted_tags

In [28]:
actual = []
predicted = []

for sentence in test_data:

    words = [word for word, tag in sentence]

    true_tags = [tag for word, tag in sentence]

    pred_tags = viterbi(words)

    actual.extend(true_tags)

    predicted.extend(pred_tags)

print("Prediction Completed!")

Prediction Completed!


In [29]:
accuracy = accuracy_score(actual, predicted)

print("Accuracy :", accuracy)

Accuracy : 1.0


In [30]:
precision, recall, f1, _ = precision_recall_fscore_support(
    actual,
    predicted,
    average="weighted",
    zero_division=0
)

print("Precision :", precision)
print("Recall :", recall)
print("F1 Score :", f1)

Precision : 1.0
Recall : 1.0
F1 Score : 1.0


In [31]:
print("\nClassification Report\n")

print(classification_report(
    actual,
    predicted,
    zero_division=0
))


Classification Report

              precision    recall  f1-score   support

          DT       1.00      1.00      1.00       200
          JJ       1.00      1.00      1.00       200
          NN       1.00      1.00      1.00       200
          RB       1.00      1.00      1.00       200
         VBZ       1.00      1.00      1.00       200

    accuracy                           1.00      1000
   macro avg       1.00      1.00      1.00      1000
weighted avg       1.00      1.00      1.00      1000



In [32]:
test_sentences = [

    ["The","big","dog","runs","quickly"],

    ["A","smart","student","studies","well"],

    ["This","beautiful","bird","flies","happily"],

    ["That","small","cat","eats","slowly"],

    ["An","fast","car","drives","today"]

]

for sentence in test_sentences:

    tags = viterbi(sentence)

    print("\nSentence :", " ".join(sentence))

    print("Predicted Tags :", tags)


Sentence : The big dog runs quickly
Predicted Tags : ['DT', 'JJ', 'NN', 'VBZ', 'RB']

Sentence : A smart student studies well
Predicted Tags : ['DT', 'JJ', 'NN', 'VBZ', 'RB']

Sentence : This beautiful bird flies happily
Predicted Tags : ['DT', 'JJ', 'NN', 'VBZ', 'RB']

Sentence : That small cat eats slowly
Predicted Tags : ['DT', 'JJ', 'NN', 'VBZ', 'RB']

Sentence : An fast car drives today
Predicted Tags : ['DT', 'JJ', 'NN', 'VBZ', 'RB']


In [33]:
print("\nAnalysis")
print("-"*50)

print("1. The HMM predicts POS tags using Initial, Transition and Emission probabilities.")

print("2. Laplace smoothing avoids zero probabilities.")

print("3. Log-space computation prevents numerical underflow.")

print("4. Unknown words receive a very small emission probability.")

print("5. The model predicts the most probable POS tag sequence using the Vectorized Viterbi Algorithm.")

print("6. Accuracy, Precision, Recall and F1-score evaluate the performance of the model.")


Analysis
--------------------------------------------------
1. The HMM predicts POS tags using Initial, Transition and Emission probabilities.
2. Laplace smoothing avoids zero probabilities.
3. Log-space computation prevents numerical underflow.
4. Unknown words receive a very small emission probability.
5. The model predicts the most probable POS tag sequence using the Vectorized Viterbi Algorithm.
6. Accuracy, Precision, Recall and F1-score evaluate the performance of the model.
